In [1]:
%load_ext autotime

import scarf

scarf.__version__

'1.0.0'

time: 2.02 s (started: 2026-07-02 22:15:35 +02:00)


In [2]:
scarf.fetch_dataset(
    dataset_name="kang_15K_pbmc_rnaseq", save_path="scarf_datasets", as_zarr=True
)

scarf.fetch_dataset(
    dataset_name="kang_14K_ifnb-pbmc_rnaseq", save_path="scarf_datasets", as_zarr=True
)

time: 30.8 s (started: 2026-07-02 22:15:37 +02:00)


In [3]:
ds_ctrl = scarf.DataStore("scarf_datasets/kang_15K_pbmc_rnaseq/data.zarr/", nthreads=4)

ds_ctrl

DataStore has 8487 (14619) cells with 1 assays: RNA
   Cell metadata:
            'I', 'ids', 'names', 'cluster_labels', 'RNA_percentMito', 
            'RNA_nCounts', 'RNA_leiden_cluster', 'RNA_nFeatures', 'RNA_percentRibo', 'RNA_UMAP2', 
            'RNA_UMAP1'
   RNA assay has 11352 (35635) features and following metadata:
            'I', 'ids', 'names', 'nCells', 'I__hvgs', 
            'dropOuts'

time: 448 ms (started: 2026-07-02 22:16:08 +02:00)


In [4]:
ds_stim = scarf.DataStore(
    "scarf_datasets/kang_14K_ifnb-pbmc_rnaseq/data.zarr", nthreads=4
)

ds_stim

DataStore has 10111 (14446) cells with 1 assays: RNA
   Cell metadata:
            'I', 'ids', 'names', 'cluster_labels', 'RNA_percentMito', 
            'RNA_percentRibo', 'RNA_nFeatures', 'RNA_leiden_cluster', 'RNA_nCounts', 'RNA_UMAP1', 
            'RNA_UMAP2'
   RNA assay has 11051 (35635) features and following metadata:
            'I', 'ids', 'names', 'I__hvgs', 'dropOuts', 
            'nCells'

time: 479 ms (started: 2026-07-02 22:16:08 +02:00)


In [5]:
scarf.AssayMerge(
    zarr_path="scarf_datasets/kang_merged_pbmc_rnaseq.zarr",
    assays=[ds_ctrl.RNA, ds_stim.RNA],
    names=["ctrl", "stim"],
    merge_assay_name="RNA",
    overwrite=True,
).dump()

ValueError: Codec does not support buffers of > 2147483647 bytes

time: 12.5 s (started: 2026-07-02 22:16:09 +02:00)


In [6]:
ds = scarf.DataStore("scarf_datasets/kang_merged_pbmc_rnaseq.zarr", nthreads=4)

Minimum cell count (0) is lower than size factor multiplier (1000)



Percentage feature RNA_percentMito not added because not detected in any cell



Percentage feature RNA_percentRibo not added because not detected in any cell



More than of half of the less have less than 10 features for assay: RNA. Will not remove low quality cells automatically.



time: 4.94 s (started: 2026-07-02 22:16:21 +02:00)


In [7]:
ds

DataStore has 29065 (29065) cells with 1 assays: RNA
   Cell metadata:
            'I', 'ids', 'names', 'orig_RNA_UMAP1', 'orig_RNA_percentRibo', 
            'orig_RNA_percentMito', 'orig_RNA_UMAP2', 'orig_RNA_nCounts', 'RNA_nFeatures', 'orig_RNA_nFeatures', 
            'orig_cluster_labels', 'orig_RNA_leiden_cluster', 'RNA_nCounts'
   RNA assay has 0 (35635) features and following metadata:
            'I', 'ids', 'names', 'nCells', 'dropOuts', 
          

time: 63.7 ms (started: 2026-07-02 22:16:26 +02:00)


In [8]:
ds.cells.head()

,I,ids,names,orig_RNA_UMAP1,orig_RNA_nCounts,orig_RNA_percentRibo,orig_RNA_UMAP2,orig_RNA_percentMito,RNA_nFeatures,orig_cluster_labels,orig_RNA_nFeatures,RNA_nCounts,orig_RNA_leiden_cluster
0,True,ctrl__TGTTAAGACACTGA-1,TGTTAAGACACTGA-1,NaN,574.0,20.034843,NaN,0.000000,0.0,nan,304.0,0.0,-1
1,True,ctrl__TCACCCGAGCTATG-1,TCACCCGAGCTATG-1,NaN,1682.0,0.059453,NaN,0.000000,0.0,nan,62.0,0.0,-1
2,True,ctrl__TCACCCGAACCACA-1,TCACCCGAACCACA-1,-1.768571,1168.0,31.763699,-5.990341,0.171233,0.0,CD4 Memory T,434.0,0.0,1
3,True,ctrl__TGTAGTCTACCAGT-1,TGTAGTCTACCAGT-1,-3.052445,2158.0,32.483781,-12.830768,0.185357,0.0,CD8 T,837.0,0.0,7
4,True,ctrl__TATCAGCTAGTAGA-1,TATCAGCTAGTAGA-1,-12.654625,1395.0,34.695341,-5.425503,0.143369,0.0,B,592.0,0.0,5


time: 202 ms (started: 2026-07-02 22:16:26 +02:00)


In [9]:
ds.cells.insert(
    column_name="sample_id",
    values=[x.split("__")[0] for x in ds.cells.fetch_all("ids")],
    overwrite=True,
)

time: 51.5 ms (started: 2026-07-02 22:16:26 +02:00)


In [10]:
ctrl_labels = list(ds_ctrl.cells.fetch_all("cluster_labels"))
stim_labels = list(ds_stim.cells.fetch_all("cluster_labels"))

ds.cells.insert(
    column_name="imported_labels", values=ctrl_labels + stim_labels, overwrite=True
)

time: 72.8 ms (started: 2026-07-02 22:16:26 +02:00)


In [11]:
ctrl_valid_cells = list(ds_ctrl.cells.fetch_all("I"))
stim_valid_cells = list(ds_stim.cells.fetch_all("I"))

ds.cells.update_key(values=ctrl_valid_cells + stim_valid_cells, key="I")

time: 64.8 ms (started: 2026-07-02 22:16:26 +02:00)


In [12]:
ds.cells.to_pandas_dataframe(["sample_id"], key="I")["sample_id"].value_counts()

sample_id
ctrl    9441
stim    9157
Name: count, dtype: int64

time: 58.8 ms (started: 2026-07-02 22:16:27 +02:00)


In [13]:
ds.mark_hvgs(min_cells=10, top_n=2000, min_mean=-3, max_mean=2, max_var=6)

Calculating summary statistics



IndexError: index 0 is out of bounds for axis 0 with size 0

time: 659 ms (started: 2026-07-02 22:16:27 +02:00)


In [14]:
ds.make_graph(feat_key="hvgs", k=21, dims=25, n_centroids=100)

make_graph step 1/7: normalize expression matrix (computing)



make_graph step 1/7: normalize expression matrix finished in 0.0s (computing)



ValueError: ERROR: Either I__hvgs does not exist or is not bool type

time: 186 ms (started: 2026-07-02 22:16:27 +02:00)


In [15]:
ds.run_umap(n_epochs=250, spread=5, min_dist=1, parallel=True)

KeyError: 'latest_feat_key'

time: 79.3 ms (started: 2026-07-02 22:16:27 +02:00)


In [16]:
ds.cells.head()

,I,ids,names,orig_RNA_percentMito,orig_RNA_percentRibo,orig_RNA_nCounts,orig_RNA_UMAP1,orig_cluster_labels,orig_RNA_UMAP2,RNA_nFeatures,RNA_nCounts,imported_labels,orig_RNA_nFeatures,orig_RNA_leiden_cluster,sample_id
0,True,ctrl__TGTTAAGACACTGA-1,TGTTAAGACACTGA-1,0.000000,20.034843,574.0,NaN,nan,NaN,0.0,0.0,CD4 naive T,304.0,-1,ctrl
1,True,ctrl__TCACCCGAGCTATG-1,TCACCCGAGCTATG-1,0.000000,0.059453,1682.0,NaN,nan,NaN,0.0,0.0,CD 14 Mono,62.0,-1,ctrl
2,True,ctrl__TCACCCGAACCACA-1,TCACCCGAACCACA-1,0.171233,31.763699,1168.0,-1.768571,CD4 Memory T,-5.990341,0.0,0.0,CD 14 Mono,434.0,1,ctrl
3,True,ctrl__TGTAGTCTACCAGT-1,TGTAGTCTACCAGT-1,0.185357,32.483781,2158.0,-3.052445,CD8 T,-12.830768,0.0,0.0,CD 14 Mono,837.0,7,ctrl
4,False,ctrl__TATCAGCTAGTAGA-1,TATCAGCTAGTAGA-1,0.143369,34.695341,1395.0,-12.654625,B,-5.425503,0.0,0.0,nan,592.0,5,ctrl


time: 325 ms (started: 2026-07-02 22:16:28 +02:00)


In [17]:
ds.plot_layout(
    layout_key="RNA_UMAP", color_by="sample_id", cmap="RdBu", legend_ondata=False
)

KeyError: 'RNA_UMAP1 does not exist in the metadata columns.'

time: 500 ms (started: 2026-07-02 22:16:28 +02:00)


In [18]:
ds.plot_layout(layout_key="RNA_UMAP", color_by="imported_labels", legend_ondata=False)

KeyError: 'RNA_UMAP1 does not exist in the metadata columns.'

time: 97.9 ms (started: 2026-07-02 22:16:28 +02:00)


In [19]:
ds.cells.insert(
    column_name="is_ctrl",
    values=(ds.cells.fetch_all("sample_id") == "ctrl"),
    overwrite=True,
)

time: 54.1 ms (started: 2026-07-02 22:16:28 +02:00)


In [20]:
ds.make_graph(feat_key="hvgs", k=21, dims=25, n_centroids=100, pca_cell_key="is_ctrl")

make_graph step 1/7: normalize expression matrix (computing)



make_graph step 1/7: normalize expression matrix finished in 0.1s (computing)



ValueError: ERROR: Either I__hvgs does not exist or is not bool type

time: 159 ms (started: 2026-07-02 22:16:29 +02:00)


In [21]:
ds.run_umap(n_epochs=250, spread=5, min_dist=1, parallel=True, label="pUMAP")

KeyError: 'latest_feat_key'

time: 65 ms (started: 2026-07-02 22:16:29 +02:00)


In [22]:
ds.plot_layout(
    layout_key="RNA_pUMAP", color_by="sample_id", cmap="RdBu", legend_ondata=False
)

KeyError: 'RNA_pUMAP1 does not exist in the metadata columns.'

time: 111 ms (started: 2026-07-02 22:16:29 +02:00)


In [23]:
ds.plot_layout(layout_key="RNA_pUMAP", color_by="imported_labels", legend_ondata=False)

KeyError: 'RNA_pUMAP1 does not exist in the metadata columns.'

time: 98.3 ms (started: 2026-07-02 22:16:29 +02:00)


In [24]:
ds.make_graph(
    feat_key="hvgs",
    k=21,
    dims=25,
    n_centroids=100,
    harmonize=True,
    batch_columns=["sample_id"],
)

ds.run_umap(n_epochs=250, spread=5, min_dist=1, parallel=True, label="hUMAP")

ds.run_leiden_clustering(resolution=1.0)

make_graph step 1/7: normalize expression matrix (computing)



make_graph step 1/7: normalize expression matrix finished in 0.1s (computing)



ValueError: ERROR: Either I__hvgs does not exist or is not bool type

time: 221 ms (started: 2026-07-02 22:16:29 +02:00)


In [25]:
ds.plot_layout(
    layout_key="RNA_hUMAP", color_by="sample_id", cmap="RdBu", legend_ondata=False
)

ds.plot_layout(layout_key="RNA_hUMAP", color_by="imported_labels", legend_ondata=False)

KeyError: 'RNA_hUMAP1 does not exist in the metadata columns.'

time: 108 ms (started: 2026-07-02 22:16:29 +02:00)


In [26]:
ds.metric_lisi(
    label_colnames=["sample_id", "imported_labels"],
    save_result=True,
)

ds.metric_silhouette(res_label="RNA_leiden_cluster")

ds.metric_integration(batch_labels=["sample_id", "imported_labels"], metric="ari")

KeyError: 'latest_cell_key'

time: 66.8 ms (started: 2026-07-02 22:16:29 +02:00)
